# Phase 10.5E: Final Out-of-Time Benchmark

This notebook analyzes the definitive out-of-time test benchmark on the frozen **9,311-sample test partition** (`2025-06-07 to 2026-08-28 UTC`).

### Predefined Benchmark Models:
1. **Naive Persistence Baseline** ($y_t \to y_{t+h}$)
2. **Ridge Regression v1** (64 features, Pollutants-only)
3. **Random Forest v1** (64 features, Pollutants-only)
4. **TensorFlow Feed-Forward DNN v1** (64 features, Pollutants-only)
5. **Ridge Regression v2 (EXP-005)** (114 features, Weather-Enriched)
6. **Hybrid AQI Specialist (EXP-017)** (LightGBM $h1-6$ + Ridge $h7-72$, 114 features)
7. **Persistence-Aware Hybrid (EXP-019)** (LightGBM $h1-6$ + Ridge $h7-37$ + Blended Persistence $h38-72$, 114 features)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

df_benchmark = pd.read_csv('../data/models/final_test_benchmark.csv')
with open('../data/models/final_test_benchmark.json', 'r') as f:
    benchmark_json = json.load(f)

df_benchmark

## 1. Overall Test Metric Comparison (RMSE, MAE, R²)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. RMSE
sns.barplot(data=df_benchmark, y="Model Name", x="Overall RMSE", ax=axes[0], palette="crest_r")
axes[0].set_title("Overall Test RMSE (Lower is Better)", fontweight="bold")
axes[0].set_xlim(70, 95)

# 2. MAE
sns.barplot(data=df_benchmark, y="Model Name", x="Overall MAE", ax=axes[1], palette="viridis_r")
axes[1].set_title("Overall Test MAE (Lower is Better)", fontweight="bold")
axes[1].set_ylabel("")
axes[1].set_yticklabels([])

# 3. R²
sns.barplot(data=df_benchmark, y="Model Name", x="Overall R²", ax=axes[2], palette="magma")
axes[2].set_title("Overall Test R² (Higher is Better)", fontweight="bold")
axes[2].set_ylabel("")
axes[2].set_yticklabels([])
axes[2].set_xlim(0.2, 0.55)

plt.tight_layout()
plt.show()

## 2. All 72 Prediction Horizon Error Curves (h=1 to h=72)

In [ ]:
plt.figure(figsize=(14, 7))
horizons = np.arange(1, 73)

colors = {
    "persistence_aware_hybrid_exp019": ("#2ca02c", 2.5, "-"),
    "hybrid_specialist_exp017": ("#1f77b4", 2.0, "-"),
    "ridge_v2_weather": ("#ff7f0e", 1.8, "--"),
    "ridge_v1_pollutants": ("#9467bd", 1.5, ":"),
    "naive_persistence": ("#d62728", 1.5, "-."),
    "tensorflow_dnn_v1": ("#8c564b", 1.5, ":"),
    "random_forest_v1": ("#7f7f7f", 1.2, ":"),
}

for key, item in benchmark_json["models"].items():
    if key in colors:
        c, lw, ls = colors[key]
        rmse_curve = item["metrics"]["per_horizon_rmse"]
        plt.plot(horizons, rmse_curve, label=item["name"], color=c, linewidth=lw, linestyle=ls)

plt.title("72-Hour Prediction Horizon Error Profiles (Out-of-Time Test Set)", fontsize=14, fontweight="bold")
plt.xlabel("Forecast Horizon (Hours Ahead)", fontsize=12)
plt.ylabel("Test RMSE (AQI Points)", fontsize=12)
plt.xticks(np.arange(0, 75, 6))
plt.legend(loc="upper left", fontsize=10)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## 3. Core Scientific Research Findings

| Core Question | Empirical Evidence & Conclusion |
| :--- | :--- |
| **Did weather telemetry help?** | **YES (Decisive)**. Ridge v1 (Pollutants-only) $\to$ Ridge v2 (Weather-Enriched) reduced overall RMSE from **82.97 to 78.38** (**+5.53% overall error reduction**, $+20.8\%$ relative $R^2$ increase). |
| **Does hybrid specialization generalize?** | **YES**. LightGBM on short horizons ($h1-6$) cuts immediate $h+1$ RMSE to **50.43** (beating Ridge by 7.3% and Naive by 26.9%). |
| **Does persistence remain strong at long horizons?** | **YES**. Combining Ridge with long-horizon linear persistence blending (**EXP-019**) delivers the #1 test performance across all 72 horizons (**75.91 overall RMSE**, **$R^2 = 0.4858$**, and $h+72$ RMSE = **77.43**). |
| **Is the hybrid worth production complexity?** | **YES**. EXP-019 beats Naive by **11.06%**, beats Ridge v1 by **8.51%**, and beats Ridge v2 by **3.15%**, providing the safest extreme-event RMSE (**142.65**). |